# End of week 1 exercise

Build a tool that takes a technical question and responds with an explanation, using the Frontier APIs and Ollama running locally.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
for folder in [Path.cwd(), *Path.cwd().parents]:
    env_file = folder / ".env"
    if env_file.exists():
        load_dotenv(env_file, override=True)
        break

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
anthropic_url = "https://api.anthropic.com/v1/"
ollama_url = "http://localhost:11434/v1"

openai = OpenAI()
gemini = OpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url=gemini_url)
anthropic = OpenAI(api_key=os.getenv("ANTHROPIC_API_KEY"), base_url=anthropic_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

MODEL_GPT = "gpt-4o-mini"
MODEL_GEMINI = "gemini-3.1-flash-lite"
MODEL_CLAUDE = "claude-haiku-4-5"
MODEL_OLLAMA = "qwen2.5:0.5b"

In [ ]:
system_prompt = """
You are an expert software engineering tutor.
Explain the question clearly, with a short example and any common pitfalls.
Respond in markdown, and do not wrap the whole reply in a code block.
""".strip()

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
def explain(client, model):
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        stream=True,
    )
    answer = ""
    handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        answer += chunk.choices[0].delta.content or ""
        update_display(Markdown(answer), display_id=handle.display_id)

In [ ]:
explain(openai, MODEL_GPT)

In [ ]:
explain(gemini, MODEL_GEMINI)

In [ ]:
explain(anthropic, MODEL_CLAUDE)

In [ ]:
explain(ollama, MODEL_OLLAMA)